# KNN Coffee Bean Predictor
Predicts a coffee bean (`name`) from roast level, aroma, structure, body, flavour, and aftertaste.

In [25]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings('ignore')

In [26]:
df = pd.read_csv('coffee_data.csv')
print(f'Shape: {df.shape}')
print(f'Roast levels: {df["roast_level"].unique()}')
# structure has a lot of nulls - not all reviews include acidity/structure
print(f'Missing values:\n{df.isnull().sum()}')
df.head()

Shape: (8999, 8)
Roast levels: ['Light' 'Medium-Light' 'Medium' 'Medium-Dark' 'Dark' nan 'Very Dark']
Missing values:
name              0
score             7
roast_level     408
aroma           129
structure      5100
body             71
flavour          75
aftertaste      862
dtype: int64


,name,score,roast_level,aroma,structure,body,flavour,aftertaste
0,Papua New Guinea Chimbu,95.0,Light,9.0,9.0,9.0,9.0,9.0
1,Hong Kong Night,93.0,Medium-Light,9.0,NaN,8.0,9.0,8.0
2,Hong Kong Day,93.0,Medium-Light,8.0,NaN,9.0,9.0,8.0
3,Sunset Blend,92.0,Medium-Light,8.0,NaN,8.0,9.0,8.0
4,Ethiopia Sidama Yaye Natural — Special Project,96.0,Light,9.0,9.0,9.0,10.0,9.0


In [27]:
FEATURES = ['roast_level', 'aroma', 'structure', 'body', 'flavour', 'aftertaste', 'score']
TARGET = 'name'

X = df[FEATURES].copy()
y = df[TARGET]

# roast levels have a natural order so ordinal encoding makes more sense than one-hot
roast_order = [['Light', 'Medium-Light', 'Medium', 'Medium-Dark', 'Dark']]

preprocessor = ColumnTransformer(transformers=[
    ('roast', Pipeline([
        ('encode', OrdinalEncoder(categories=roast_order, handle_unknown='use_encoded_value', unknown_value=-1)),
        ('scale', StandardScaler())
    ]), ['roast_level']),
    ('numeric', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ]), ['aroma', 'structure', 'body', 'flavour', 'aftertaste', 'score'])
])

print('Preprocessor ready.')

Preprocessor ready.


In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {len(X_train)} samples | Test: {len(X_test)} samples')

Train: 7199 samples | Test: 1800 samples


In [29]:
# try k=1 to 15 and pick whichever does best on 5-fold CV
k_scores = {}
for k in range(1, 16):
    pipe = Pipeline([
        ('prep', preprocessor),
        ('knn', KNeighborsClassifier(n_neighbors=k, metric='euclidean'))
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')
    k_scores[k] = scores.mean()

best_k = max(k_scores, key=k_scores.get)
print(f'Best k={best_k}  (CV accuracy: {k_scores[best_k]:.3f})')
for k, score in k_scores.items():
    bar = '#' * int(score * 20)
    print(f'  k={k:2d}: {score:.3f} {bar}')

Best k=6  (CV accuracy: 0.004)
  k= 1: 0.003 
  k= 2: 0.003 
  k= 3: 0.002 
  k= 4: 0.003 
  k= 5: 0.003 
  k= 6: 0.004 
  k= 7: 0.004 
  k= 8: 0.003 
  k= 9: 0.003 
  k=10: 0.003 
  k=11: 0.003 
  k=12: 0.003 
  k=13: 0.003 
  k=14: 0.003 
  k=15: 0.003 


In [30]:
model = Pipeline([
    ('prep', preprocessor),
    ('knn', KNeighborsClassifier(n_neighbors=best_k, metric='euclidean'))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f'Test accuracy (exact match): {accuracy_score(y_test, y_pred):.3f}')

Test accuracy (exact match): 0.005


In [31]:
# exact match will be low since hundreds of beans share similar scores
# top-N tells us whether the right answer is at least in the shortlist
X_test_transformed = model.named_steps['prep'].transform(X_test)
knn = model.named_steps['knn']
classes = knn.classes_
proba = knn.predict_proba(X_test_transformed)

top3, top5 = 0, 0
for i, true_label in enumerate(y_test.values):
    top_idx = np.argsort(proba[i])[::-1]
    if true_label in classes[top_idx[:3]]: top3 += 1
    if true_label in classes[top_idx[:5]]: top5 += 1

n = len(y_test)
print(f'Top-1 accuracy: {accuracy_score(y_test, y_pred):.3f}')
print(f'Top-3 accuracy: {top3/n:.3f}')
print(f'Top-5 accuracy: {top5/n:.3f}')

Top-1 accuracy: 0.005
Top-3 accuracy: 0.004
Top-5 accuracy: 0.009


In [32]:
def predict_coffee(roast_level, aroma, structure, body, flavour, aftertaste, score=95, top_n=5):
    input_df = pd.DataFrame([{
        'roast_level': roast_level,
        'aroma':       aroma,
        'structure':   structure,
        'body':        body,
        'flavour':     flavour,
        'aftertaste':  aftertaste,
        'score':       score
    }])

    X_in = model.named_steps['prep'].transform(input_df)
    p = knn.predict_proba(X_in)[0]
    top_idx = np.argsort(p)[::-1][:top_n]

    print(f'Attributes: roast={roast_level}, aroma={aroma}, structure={structure}, body={body}, flavour={flavour}, aftertaste={aftertaste}, score={score}')
    print()
    print(f'Top {top_n} predicted coffees:')
    for rank, idx in enumerate(top_idx, 1):
        print(f'  {rank}. {classes[idx]}  ({p[idx]:.1%})')


# pass None for structure if you don't have it - the imputer handles it
# lower score to e.g. 90 to include more mid-range beans in the neighbourhood
predict_coffee('Light', aroma=9, structure=9, body=9, flavour=10, aftertaste=9)

Attributes: roast=Light, aroma=9, structure=9, body=9, flavour=10, aftertaste=9, score=95

Top 5 predicted coffees:
  1. Wilton Benitez Colombia Yellow Bourbon  (16.7%)
  2. Taiwan Natural Alishan Lalauya Zou Zhu Yuan Gesha  (16.7%)
  3. Panama Kaizen Lot GW-01  (16.7%)
  4. Panama Elida Natural Lot #13  (16.7%)
  5. Wilton Benitez Geisha  (16.7%)
